In [17]:
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import cv2
import torch
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator


In [18]:
base_path = '../assets'
image_path = f"{base_path}/37_angelica_left_14.jpeg" # 41_daniel_right_1.jpeg
yolo_model_path = '../runs/detect/train2/weights/best.pt'
sam_model_path = '../models/sam_vit_h_4b8939.pth'

In [19]:
def plot_pred_resutls(inf_results):
    plot_resutls = inf_results[0].plot()
    inf_image = cv2.cvtColor(plot_resutls, cv2.COLOR_BGR2RGB)
    plt.imshow(inf_image)
    plt.axis('on')
    plt.show()

def get_best_detections(results):
    # Inicializar variables para guardar las mejores detecciones
    best_coin = None  # (x_center, y_center, score)
    best_foot = None  # (x_center, y_center, score)

    # Iterar sobre los resultados de las detecciones
    for box in results[0].boxes:
        # Extraer la clase, puntuación y coordenadas del cuadro delimitador
        cls = int(box.cls.numpy().item())  # Clase del objeto (0 para coin, 1 para foot, según data.yaml)
        conf = float(box.conf.numpy().item())  # Puntaje de confianza
        x1, y1, x2, y2 = box.xyxy.numpy()[0]  # Coordenadas del cuadro delimitador

        # Calcular el centro (x, y) del cuadro delimitador
        x_center = (x1 + x2) / 2
        y_center = (y1 + y2) / 2

        # Clasificar detecciones por clase
        if cls == 0:  # Clase "coin"
            if best_coin is None or conf > best_coin[2]:
                best_coin = (x_center, y_center, conf)  # Guardar mejor detección para "coin"
        elif cls == 1:  # Clase "foot"
            if best_foot is None or conf > best_foot[2]:
                best_foot = (x_center, y_center, conf)  # Guardar mejor detección para "foot"

    # Crear un diccionario con los resultados
    return {
        "coin": {"center": (best_coin[0], best_coin[1]), "score": best_coin[2]} if best_coin else None,
        "foot": {"center": (best_foot[0], best_foot[1]), "score": best_foot[2]} if best_foot else None
    }

def get_object_properties(mask):
    # Encuentra los índices donde el valor es True
    indices = np.argwhere(mask)

    # Si la máscara contiene valores True, calcula las propiedades
    if indices.size > 0:
        y_min, x_min = indices.min(axis=0)  # Coordenadas superiores izquierda
        y_max, x_max = indices.max(axis=0)  # Coordenadas inferiores derecha

        coord_x = x_min
        coord_y = y_min
        width = x_max - x_min + 1  # +1 para incluir el tamaño total
        height = y_max - y_min + 1  # +1 para incluir el tamaño total

        return coord_x, coord_y, width, height
    else:
        # Si no hay valores True, retorna None o valores por defecto
        return None, None, 0, 0

def get_gpu_available(machine='mac-silicon'):
    if machine == 'colab':
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    elif machine == 'mac-silicon':
        return torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')

get_gpu_available()

device(type='mps')

In [20]:
device = get_gpu_available()
sam = sam_model_registry['vit_h'](checkpoint=sam_model_path).to(device=device)
mask_generator = SamAutomaticMaskGenerator(sam) 

In [21]:

def process_image(image_path):
    image = plt.imread(image_path)
    custom_model = YOLO(yolo_model_path)

    inf_results = custom_model(image_path)

    height, width = image.shape[:2]
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plot_resutls = inf_results[0].plot()
    labeled_image = cv2.cvtColor(plot_resutls, cv2.COLOR_BGR2RGB)

    detections = get_best_detections(inf_results)
    coin_center_x = int(detections['coin']['center'][0])
    coin_center_y = int(detections['coin']['center'][1])
    foot_center_x = int(detections['foot']['center'][0])
    foot_center_y = int(detections['foot']['center'][1])

    predictor = SamPredictor(sam)
    predictor.set_image(image_rgb)

    coin_input_point = np.array([[coin_center_x, coin_center_y]])
    coin_input_label = np.array([1])
    foot_input_point = np.array([[foot_center_x, foot_center_y]]) 
    foot_input_label = np.array([1])

    coin_masks, coin_scores, coin_logits = predictor.predict(
        point_coords=coin_input_point,
        point_labels=coin_input_label,
        multimask_output=True
    )
    coin_masks_dict =[ {'mask': m, 'score': s} for m, s in zip(coin_masks, coin_scores)]
    coin_masks_dict.sort(reverse=True, key=lambda x: x['score'])
    coin_best_mask = coin_masks_dict[0]

    # foot
    foot_masks, foot_scores, foot_logits = predictor.predict(
        point_coords=foot_input_point,
        point_labels=foot_input_label,
        multimask_output=True
    )
    foot_masks_dict =[ {'mask': m, 'score': s} for m, s in zip(foot_masks, foot_scores)]
    foot_masks_dict.sort(reverse=True, key=lambda x: x['score'])
    foot_best_mask = foot_masks_dict[0]

    coin_x, coin_y, coin_w, coin_h = get_object_properties(coin_best_mask['mask'])
    foot_x, foot_y, foot_w, foot_h = get_object_properties(foot_best_mask['mask'])
    image_with_box = image_rgb.copy()
    cv2.rectangle(image_with_box, (coin_x, coin_y), (coin_x + coin_w, coin_y + coin_h), (255, 0, 0), 2)
    cv2.rectangle(image_with_box, (foot_x, foot_y), (foot_x + foot_w, foot_y + foot_h), (0, 255, 0), 2)
    
    image_with_box = image_rgb.copy()
    cv2.rectangle(image_with_box, (coin_x, coin_y), (coin_x + coin_w, coin_y + coin_h), (255, 0, 0), 2)
    cv2.rectangle(image_with_box, (foot_x, foot_y), (foot_x + foot_w, foot_y + foot_h), (0, 255, 0), 2)
    
    labeled_image = cv2.cvtColor(labeled_image, cv2.COLOR_RGB2BGR)
    image_with_box = cv2.cvtColor(image_with_box, cv2.COLOR_RGB2BGR)

    data = {
        #"img_name": image_path.split('/')[-1], 
        "width": width, 
        "height": height,
        "coin_width": coin_w,
        "coin_height": coin_h,
        "foot_width": foot_w
    }

    return  data, labeled_image, image_with_box

In [22]:
data, labeled_image, image_with_box = process_image(image_path)
data


image 1/1 /Users/gipo/my-stuff/MIT/shoe-sizer/experiments/../assets/37_angelica_left_14.jpeg: 1152x1504 1 coin, 2 foots, 150.3ms
Speed: 8.0ms preprocess, 150.3ms inference, 4.8ms postprocess per image at shape (1, 3, 1152, 1504)


{'width': 1600,
 'height': 1200,
 'coin_width': 146,
 'coin_height': 149,
 'foot_width': 1217}

In [23]:
# plt.imshow(labeled_image)

In [24]:
# plt.imshow(image_with_box)

In [25]:
import joblib
best_model_name = 'shoe_sizer_model'
best_model_path = f'../models/{best_model_name}.pkl'

loaded_model = joblib.load(best_model_path)

In [26]:
import pandas as pd

data['sex_M'] = 0
data['foot_side_right'] = 0

data_df = pd.DataFrame(data, index=[0])
predicted_size_cm = loaded_model.predict(data_df)

print(f"Predicción: {predicted_size_cm[0]:.2f} cm")

Predicción: 24.26 cm


In [27]:
%pip install --upgrade gradio -q

Note: you may need to restart the kernel to use updated packages.


In [28]:
import gradio as gr

def compute_size_eu(size_cm):
    correction_factor = 1
    talla_baja = round(size_cm*3/2 + 1,0)
    ideal_size_eu = round(size_cm*3/2 + 2,0)
    talla_superior = round(size_cm*3/2 + 3,0)
    return ideal_size_eu - correction_factor


def process_raw_image(image, sex, foot_side):

    data, labeled_image, image_with_box = process_image(image)
    data['sex_M'] = 1 if sex == 'M' else 0
    data['foot_side_right'] = 1 if foot_side == 'right' else 0
    data_df = pd.DataFrame(data, index=[0])
    predicted_size_cm = loaded_model.predict(data_df)

    return f"Foot size prediction: {predicted_size_cm[0]:.2f} cm. The ideal size in EU is {compute_size_eu(predicted_size_cm[0])}"


In [29]:
interface = gr.Interface(
    fn=process_raw_image,  # Función que se ejecutará con los inputs
    inputs=[
        gr.Image(type="filepath", label="Load the image"),
        gr.Radio(["M", "F"], label="Gender"),
        gr.Radio(["left", "right"], label="Foot side")
    ],
    outputs=gr.Textbox(label="Results"),
    title="Shoe Sizer",
    description="Cargue una imagen del pie e indique el sexo y el lado del pie para procesar."
)

# Lanzar la aplicación
interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://001425e658d5e0743c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



image 1/1 /private/var/folders/64/m87yf3yx7zngv12wrzgqgj_w0000gn/T/gradio/108c3dc095bd4631850e78d0451e1025f85c2584f4243377ce69f7fa0db779b1/41_linford_left_1.jpeg: 1152x1504 1 coin, 1 foot, 229.5ms
Speed: 17.2ms preprocess, 229.5ms inference, 3.1ms postprocess per image at shape (1, 3, 1152, 1504)

image 1/1 /private/var/folders/64/m87yf3yx7zngv12wrzgqgj_w0000gn/T/gradio/108c3dc095bd4631850e78d0451e1025f85c2584f4243377ce69f7fa0db779b1/41_linford_left_1.jpeg: 1152x1504 1 coin, 1 foot, 293.1ms
Speed: 12.0ms preprocess, 293.1ms inference, 1.7ms postprocess per image at shape (1, 3, 1152, 1504)
